In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from scipy.optimize import minimize

from rich import print
import seaborn as sns
sns.set()

import ewatercycle
import ewatercycle.models
import ewatercycle.forcing
from scipy.stats import qmc
from ipywidgets import IntProgress

In [2]:
calibration_start_date = "1975-01-01"
calibration_end_date = "2010-12-31"

In [3]:
forcing_path_ERA5 = Path.home() / "BEP-beau/BEP/code" / "CatchmentArea" / "ERA5_2" / "own_shapefile_2"
load_location = forcing_path_ERA5 / "work" / "diagnostic" / "script"  
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=load_location)

data = pd.read_csv("mohembo_daily_water_discharge_data.csv", index_col='date', parse_dates=True, dayfirst=True)
data_daily = data.resample('D').interpolate()
data_daily.columns = ['Discharge (m^3/s)']

In [5]:
Area_km2 = 173696.852

def mmday_to_m3s(mmday_data, area):
    return (mmday_data * area) / 86.4

In [6]:
def objective(modelled, observed, start, end, w_kge, w_lognse):
    start = pd.to_datetime(start).tz_localize(None)
    end = pd.to_datetime(end).tz_localize(None)
    
    modelled.index = pd.to_datetime(modelled.index)
    observed.index = pd.to_datetime(observed.index)

    df = pd.concat([modelled.reindex(observed.index, method='ffill'), observed], axis=1, keys=['Modelled', 'Observed'])
    df = df.dropna()
    df = df[(df.index > start) & (df.index < end)]

    #KGE
    r = np.corrcoef(df['Observed'], df['Modelled'])[0, 1]
    beta = np.mean(df['Modelled']) / np.mean(df['Observed'])
    CV_modelled = np.std(df['Modelled']) / np.mean(df['Modelled'])
    CV_observed = np.std(df['Observed']) / np.mean(df['Observed'])                                 
    gamma = CV_modelled / CV_observed
    
    kge = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)

    #logNSE
    a = (np.log(df['Modelled']) - np.log(df['Observed']))**2
    a1 = a.sum() 
    b = (np.log(df['Observed']) - np.log(np.mean(df['Observed'])))**2
    b1 = b.sum()
    lognse = 1 - (a1/b1)

    score = w_kge * kge + w_lognse * lognse
    return score

In [21]:
N = 2000
s_0 = np.array([0,  100,  0,  5,  0])

param_names = ["Imax", "Ce", "Sumax", "Beta", "Pmax", "Tlag", "Kf", "Ks", "FM"]
param_mins = np.array([0, 0.2, 40, 0.5, 0.001, 1, 0.01, 0.0001, 0.01])
param_maxs = np.array([8, 1, 800, 4, 0.3, 10, 0.1, 0.01, 1])

sampler = qmc.LatinHypercube(d=len(param_names))
sample = sampler.random(n=N)
parameters = qmc.scale(sample, param_mins, param_maxs)

In [22]:
ensemble = []

for counter in range(N): 
    ensemble.append(ewatercycle.models.HBVLocal(forcing=ERA5_forcing))
    config_file, _ = ensemble[counter].setup(parameters = parameters[counter],  initial_storage=s_0)
    ensemble[counter].initialize(config_file)

In [23]:
f = IntProgress(min=0, max=N)
display(f)

values = []
for ensembleMember in ensemble:
    Q_m = []
    time = []
    while ensembleMember.time < ensembleMember.end_time:
        ensembleMember.update()
        discharge_this_timestep = ensembleMember.get_value("Q")
        Q_m.append(discharge_this_timestep[0])
        time.append(ensembleMember.time_as_datetime)

    Q_m = mmday_to_m3s(np.array(Q_m), Area_km2)
    discharge_dataframe = pd.DataFrame({'model output': Q_m}, index=pd.to_datetime(time))

    obj_value = objective(discharge_dataframe['model output'], data_daily['Discharge (m^3/s)'], calibration_start_date, calibration_end_date, 0.5, 0.5)
    values.append(obj_value)
    
    del Q_m, time, discharge_dataframe, obj_value
    f.value += 1
    
for ensembleMember in ensemble:
    ensembleMember.finalize()

IntProgress(value=0, max=2000)

In [25]:
best_index = np.argmax(values)
best_parameters = parameters[best_index]
best_value = values[best_index]
print(list(zip(param_names, np.round(best_parameters, decimals=3))))
print(f"Best value: {best_value:.3f}")
best_parameters

[
    ('Imax', 1.46),
    ('Ce', 0.349),
    ('Sumax', 688.56),
    ('Beta', 3.827),
    ('Pmax', 0.29),
    ('Tlag', 2.606),
    ('Kf', 0.016),
    ('Ks', 0.008),
    ('FM', 0.895)
]

Best value: 0.718

array([1.46021786e+00, 3.48870515e-01, 6.88560472e+02, 3.82660027e+00,
       2.89658094e-01, 2.60605631e+00, 1.55160449e-02, 8.45162498e-03,
       8.95249330e-01])